# ChronoNode Python SDK Interactive Demo

This Jupyter Notebook demonstrates the core capabilities of the ChronoNode Python SDK, including:
1. **Client Initialization & Node Health checks**
2. **Fetching active blockchain networks**
3. **Querying block ranges and transaction metrics**
4. **Client-side cryptographic SPV verification** (Merkle proof and Ed25519 signatures)
5. **Integration with Pandas** for statistical analysis and plotting

## Setup and Imports

First, we make sure that the parent folder is added to our system path so we can import the `chrononode` package without a prior global installation, and then import the required modules.

In [ ]:
import os
import sys
import json
import http.server
import threading
import socket
from cryptography.hazmat.primitives.asymmetric import ed25519

# Ensure the parent directory is in the path to load the local sdk package
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from chrononode import ChronoNodeClient, calculate_leaf_hash, hash_pair

## In-Memory HTTP Mock Server Setup

To make this notebook runnable completely out-of-the-box, we will spin up a background mock HTTP server that simulates the ChronoNode REST API. 
This server handles endpoints for health checks, active chains, block query ranges, and cryptographic proofs.

In [ ]:
MOCK_PROOF_DATA = {}

class DemoHTTPHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def do_GET(self):
        if self.path == "/health":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"status": "ok", "uptime_seconds": 12345}).encode('utf-8'))
        elif self.path == "/v1/chains":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps([
                {"chain_id": "baals", "display_name": "BaaLS Network"},
                {"chain_id": "bitcoin", "display_name": "Bitcoin Network"}
            ]).encode('utf-8'))
        elif self.path == "/v1/chains/baals/blocks?from=100&to=104":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps([
                {"chain_id": "baals", "height": 100, "block_hash": "0xabc100", "timestamp": 1600000100, "tx_count": 10, "event_count": 1},
                {"chain_id": "baals", "height": 101, "block_hash": "0xabc101", "timestamp": 1600000200, "tx_count": 8, "event_count": 0},
                {"chain_id": "baals", "height": 102, "block_hash": "0xabc102", "timestamp": 1600000300, "tx_count": 15, "event_count": 4},
                {"chain_id": "baals", "height": 103, "block_hash": "0xabc103", "timestamp": 1600000400, "tx_count": 12, "event_count": 2},
                {"chain_id": "baals", "height": 104, "block_hash": "0xabc104", "timestamp": 1600000500, "tx_count": 20, "event_count": 5}
            ]).encode('utf-8'))
        elif self.path == "/v1/chains/baals/proofs/block/100":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"proof": MOCK_PROOF_DATA}).encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()

def start_mock_server() -> int:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    
    server = http.server.HTTPServer(('127.0.0.1', port), DemoHTTPHandler)
    t = threading.Thread(target=server.serve_forever)
    t.daemon = True
    t.start()
    return port

def generate_mock_proof():
    global MOCK_PROOF_DATA
    priv_key = ed25519.Ed25519PrivateKey.generate()
    pub_bytes = priv_key.public_key().public_bytes_raw()
    
    block_hash = b"\xaa" * 32
    sibling_hash = b"\xbb" * 32
    
    leaf_hash = calculate_leaf_hash(
        chain_id="baals",
        height=100,
        block_hash=block_hash,
        storage_backend="ipfs",
        storage_pointer="ipfs:QmXyZ100"
    )
    
    root_hash = hash_pair(leaf_hash, sibling_hash)
    signature = priv_key.sign(root_hash)
    
    MOCK_PROOF_DATA = {
        "version": "chrononode-proof-v1",
        "chain_id": "baals",
        "height": 100,
        "block_hash": "0x" + block_hash.hex(),
        "storage_backend": "ipfs",
        "storage_pointer": "ipfs:QmXyZ100",
        "checkpoint": {
            "checkpoint_id": "checkpoint-baals-100",
            "start_height": 100,
            "end_height": 101,
            "root": "0x" + root_hash.hex(),
            "signer_pubkey": "0x" + pub_bytes.hex(),
            "signature": "0x" + signature.hex()
        },
        "proof": [
            {
                "position": "right",
                "hash": "0x" + sibling_hash.hex()
            }
        ]
    }

# Start mock server
generate_mock_proof()
port = start_mock_server()
print(f"[*] Started local demo mock server on port {port}")

## 1. Initializing the Client

Now we instantiate the `ChronoNodeClient` pointing to our local node port.

In [ ]:
client = ChronoNodeClient(f"http://127.0.0.1:{port}")
print("ChronoNode client successfully initialized!")

## 2. Querying Node Health

Verify the node is responsive and query its health stats.

In [ ]:
health = client.health()
print(json.dumps(health, indent=2))

## 3. Listing Active Blockchains

Retrieve the list of active blockchains indexed by the ChronoNode ecosystem.

In [ ]:
chains = client.list_chains()
for c in chains:
    print(f"- {c['display_name']} ({c['chain_id']})")

## 4. Querying Historical Block Ranges

We can query a specific range of blocks to retrieve transaction and event volumes.

In [ ]:
blocks = client.get_block_range("baals", 100, 104)
for b in blocks:
    print(f"Height: {b['height']} | Hash: {b['block_hash']} | Tx Count: {b['tx_count']} | Events: {b['event_count']}")

## 5. Cryptographic Proof Retrieval & Local Verification

To ensure zero-trust security, the client can fetch a Merkle Proof for a block and verify it locally against a known checkpoint signer. 
The client will:
1. Calculate the block's leaf hash using domain-separated length-prefixed formatting.
2. Traverse the Merkle tree path to reconstruct the root hash.
3. Check the Ed25519 signature of the checkpoint root using the signer's public key.

In [ ]:
proof = client.get_block_proof("baals", 100)
print(f"Retrieved block height: {proof['height']}")
print(f"Storage location: {proof['storage_pointer']} ({proof['storage_backend']})")
print(f"Checkpoint Signer Pubkey: {proof['checkpoint']['signer_pubkey']}")

# Perform local client-side SPV verification
is_valid = client.verify_proof_locally(proof)
print(f"\n>>> Verification status: {'SUCCESS (Valid Block & Proof)' if is_valid else 'FAILED'}")

## 6. Pandas Integration & Data Analysis

We convert the retrieved block list into a Pandas DataFrame to facilitate data manipulation, statistics, and analysis.

In [ ]:
try:
    from chrononode.pandas_helper import to_dataframe
    df = to_dataframe(blocks)
    print("Pandas DataFrame successfully loaded:")
    display(df)
except ImportError:
    print("Pandas is not installed. You can install it using 'pip install pandas' to view this section.")
    df = None

### Statistical Summaries

If Pandas is installed, let's run statistics on block activity (e.g. transactions and events).

In [ ]:
if df is not None:
    # Describe the dataset
    print("=== Basic Stats ===")
    print(df[['tx_count', 'event_count']].describe())
    
    # Compute correlation between txs and events
    corr = df['tx_count'].corr(df['event_count'])
    print(f"\nCorrelation between tx_count and event_count: {corr:.4f}")